# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 05 — PLN Clássico: Fundamentos de Pré-processamento
**Do PDF ao texto tratado · spaCy + NLTK · Normalização, Stopwords, Stemming vs. Lemmatization, POS e NER**

---

### Objetivos do Laboratório
1. **Extração de Texto:** Ler documentos em PDF (`introducaoml.pdf`) com `pdfplumber`.
2. **Normalização e Limpeza:** Conversão para minúsculas, remoção de caracteres espúrios via Regex.
3. **Stop Words:** Filtragem de palavras de baixa carga semântica (NLTK + lista customizada).
4. **Stemming vs. Lematização:** Contrastar corte radical bruto (`RSLPStemmer`) com redução canônica de dicionário (`spaCy`).
5. **Análise Sintática e Semântica:** Reconhecimento de classes gramaticais (**POS Tagging**) e entidades nomeadas (**NER**).
6. ⚠️ **Atenção:** Entender a virada de paradigma entre PLN Clássico e RAG com LLMs.


### 1. Instalação e Preparação do Ambiente
Execute caso ainda não possua as bibliotecas instaladas em seu ambiente:
```bash
pip install pdfplumber spacy nltk
python -m spacy download pt_core_news_sm
```


In [ ]:
import os
import re
import pdfplumber
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

# Baixando recursos essenciais do NLTK
for recurso in ['stopwords', 'rslp', 'punkt']:
    try:
        nltk.download(recurso, quiet=True)
    except Exception:
        pass

# Importando spaCy com tratamento defensivo
import spacy
try:
    nlp = spacy.load("pt_core_news_sm")
    print("Modelo spaCy pt_core_news_sm carregado com sucesso!")
except OSError:
    print("Aviso: Baixando modelo spaCy pt_core_news_sm...")
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "pt_core_news_sm"], check=True)
    nlp = spacy.load("pt_core_news_sm")

---
### 2. Leitura e Extração de Texto do PDF (`introducaoml.pdf`)

In [ ]:
def ler_pdf(caminho_pdf):
    texto = ""
    with pdfplumber.open(caminho_pdf) as leitor_pdf:
        for pagina in leitor_pdf.pages:
            texto_pagina = pagina.extract_text()
            if texto_pagina:
                texto += texto_pagina + " "
    # Remove quebras de linha substituindo por espaço
    texto = texto.replace("\n", " ").strip()
    return texto

arquivo_pdf = "introducaoml.pdf"

if os.path.exists(arquivo_pdf):
    texto_original = ler_pdf(arquivo_pdf)
    print(f"Sucesso! Tamanho do texto original: {len(texto_original)} caracteres.")
    print("\nPrimeiros 400 caracteres:")
    print(texto_original[:400] + "...")
else:
    print(f"Aviso: Arquivo {arquivo_pdf} não encontrado.")

---
### 3. Configuração das Ferramentas de PLN e Stop Words

In [ ]:
# Stop words do NLTK combinadas com stop words personalizadas
try:
    api_stop_words = set(stopwords.words('portuguese'))
except Exception:
    api_stop_words = {'de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'é', 'com', 'não', 'uma', 'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'foi', 'ao', 'ele', 'das', 'tem', 'à', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'há', 'nos', 'já', 'está', 'eu', 'também', 'só', 'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'era', 'depois', 'sem', 'mesmo', 'aos', 'ter', 'seus', 'quem', 'nas', 'me', 'esse', 'eles', 'estão', 'você', 'tinha', 'foram', 'essa', 'num', 'nem', 'suas', 'meu', 'às', 'minha', 'têm', 'numa', 'pelos', 'elas', 'havia', 'seja', 'qual', 'será', 'nós', 'tenho', 'lhe', 'deles', 'essas', 'esses', 'pelas', 'este', 'fosse', 'dele', 'tu', 'te', 'vocês', 'vos', 'lhes', 'meus', 'minhas', 'teu', 'tua', 'teus', 'tuas', 'nosso', 'nossa', 'nossos', 'nossas', 'dela', 'delas', 'esta', 'estes', 'estas', 'aquele', 'aquela', 'aqueles', 'aquelas', 'isto', 'aquilo', 'estou', 'está', 'estamos', 'estão', 'estive', 'esteve', 'estivemos', 'estiveram', 'estava', 'estávamos', 'estavam', 'estivera', 'estivéramos', 'esteja', 'estejamos', 'estejam', 'estivesse', 'estivéssemos', 'estivessem', 'estiver', 'estivermos', 'estiverem', 'hei', 'há', 'havemos', 'hão', 'houve', 'houvemos', 'houveram', 'houvera', 'houvéramos', 'haja', 'hajamos', 'hajam', 'houvesse', 'houvéssemos', 'houvessem', 'houver', 'houvermos', 'houverem', 'houverei', 'houverá', 'houveremos', 'houverão', 'houveria', 'houveríamos', 'houveriam', 'sou', 'somos', 'são', 'era', 'éramos', 'eram', 'fui', 'foi', 'fomos', 'foram', 'fora', 'fôramos', 'seja', 'sejamos', 'sejam', 'fosse', 'fôssemos', 'fossem', 'for', 'formos', 'forem', 'serei', 'será', 'seremos', 'serão', 'seria', 'seríamos', 'seriam', 'tenho', 'tem', 'temos', 'tém', 'tinha', 'tínhamos', 'tinham', 'tive', 'teve', 'tivemos', 'tiveram', 'tivera', 'tivéramos', 'tenha', 'tenhamos', 'tenham', 'tivesse', 'tivéssemos', 'tivessem', 'tiver', 'tivermos', 'tiverem', 'terei', 'terá', 'teremos', 'terão', 'teria', 'teríamos', 'teriam'}

minhas_stop_words = {'a', 'e', 'i', 'o', 'u', 'neste', 'nesta'}
stop_words = api_stop_words | minhas_stop_words

print(f"Total de Stop Words configuradas no vocabulário: {len(stop_words)}")

---
### 4. Pipeline Completo de Limpeza e Lematização

In [ ]:
def processar_texto_pln(texto):
    # 1. Normalização: conversão para minúsculas
    texto_norm = texto.lower()
    
    # 2. Remoção de ruído: caracteres que não são letras em português ou espaços
    texto_limpo = re.sub(r'[^a-záéíóúâêîôûãõç\s]', '', texto_norm)
    
    # 3. Tokenização inteligente via spaCy
    doc = nlp(texto_limpo)
    
    # 4. Lematização e remoção de stopwords/pontuação
    clean_tokens = [
        token.lemma_ for token in doc
        if token.text not in stop_words
        and not token.is_punct
        and token.text.strip() != ""
    ]
    
    # 5. Reconstrução do texto tratado
    return ' '.join(clean_tokens)

texto_tratado = processar_texto_pln(texto_original)

print(f"Tamanho ANTES: {len(texto_original)} caracteres.")
print(f"Tamanho DEPOIS: {len(texto_tratado)} caracteres.")
print(f"Redução percentual: {(1 - len(texto_tratado)/len(texto_original))*100:.1f}%")
print("\nPrimeiros 300 caracteres após tratamento:")
print(texto_tratado[:300] + "...")

---
### 5. Stemming vs. Lematização (Comparativo Prático)
Contrastamos o corte de sufixos algorítmico do **Stemming** com a forma canônica de dicionário da **Lematização**.

In [ ]:
frase_teste = "As organizações correram rapidamente para desenvolver modelos complexos."

# 1. Stemming com RSLPStemmer (NLTK)
stemmer = RSLPStemmer()
tokens_nltk = nltk.word_tokenize(frase_teste, language='portuguese')
stems = [stemmer.stem(t) for t in tokens_nltk]

# 2. Lematização com spaCy
doc_spacy = nlp(frase_teste)
lemas = [t.lemma_ for t in doc_spacy]

print("Frase Original: ", frase_teste)
print("\nStemming (NLTK):", ' '.join(stems))
print("Lematização (spaCy):", ' '.join(lemas))

**Observação Fundamental:**
- O *Stemming* gera radicais truncados e artificiais (`organizaç`, `complex`), ignorando a gramática.
- A *Lematização* recupera o infinitivo dos verbos e a forma singular canônica de substantivos (`organização`, `correr`, `complexo`). Em português, a lematização preserva muito mais sentido!


---
### 6. Análise Sintática e Semântica: POS Tagging e NER

In [ ]:
frase_entidades = "Steve Jobs fundou a Apple em 1976 na cidade de Cupertino."
doc_entidades = nlp(frase_entidades)

print("=== 1. POS TAGGING (Part-of-Speech) ===")
for token in doc_entidades:
    print(f"{token.text:12} | Tag: {token.pos_:6} | Morfologia: {spacy.explain(token.pos_)}")

print("\n=== 2. NER (Named Entity Recognition) ===")
for ent in doc_entidades.ents:
    print(f"Entidade: {ent.text:15} | Categoria: {ent.label_:8} | Significado: {spacy.explain(ent.label_)}")

---
### 7. Quadro-Resumo dos Conceitos Aplicados

| Técnica | O que faz | Exemplo Prático |
| :--- | :--- | :--- |
| **Normalização** | Padroniza minúsculas e pontuação | `"Machine"` e `"machine"` tornam-se iguais |
| **Remoção de Ruído** | Elimina números e símbolos desnecessários | `"IA em 2026!"` → `"ia em"` |
| **Remoção de Stopwords** | Retira palavras gramaticais de pouca carga semântica | `"O gato de botas"` → `"gato botas"` |
| **Lematização** | Reduz as palavras ao seu lema de dicionário | `"correram"`, `"correndo"` → `"correr"` |
| **POS Tagging** | Rotula a classe gramatical de cada palavra | `"Apple"` → `PROPN` (Nome próprio) |
| **NER** | Detecta e classifica entidades no mundo real | `"Steve Jobs"` → `PER`, `"Apple"` → `ORG` |

---
### ⚠️ Ressalva Crucial de Conexão com os Próximos Labs (Embeddings & RAG)
O pré-processamento pesado demonstrado neste laboratório (remover stop words e lematizar tudo) é a base do **PLN Clássico**, ideal para TF-IDF, buscas booleanas e algoritmos tradicionais.

**Nos próximos laboratórios (Embeddings e RAG), a regra se inverte!**
Modelos de linguagem modernos (Transformers e LLMs) dependem da sintaxe e da ordem natural das palavras para compreender nuances de significado. Portanto, para embeddings e RAG, preservamos o texto natural!
